In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
# Crear widgets
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("catalogo", "catalog_au")
dbutils.widgets.text("esquema", "bronze")
dbutils.widgets.text("storageName", "adlsfinalproject")

In [0]:
# Obtener valores de widgets
container = dbutils.widgets.get("container")
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")
storageName = dbutils.widgets.get("storageName")

# Obtener ruta de circuits.csv
ruta = f"abfss://{container}@{storageName}.dfs.core.windows.net/circuits.csv"

In [0]:
# Crear DataFrame df_circuits infiriendo el schema.
df_circuits = spark.read.option('header', True)\
                        .option('inferSchema', True)\
                        .csv(ruta)

In [0]:
# Definir esquema para el DataFrame df_circuits
circuits_schema = StructType(fields=[StructField("circuitId", IntegerType(), False),
                                     StructField("circuitRef", StringType(), True),
                                     StructField("name", StringType(), True),
                                     StructField("location", StringType(), True),
                                     StructField("country", StringType(), True),
                                     StructField("lat", DoubleType(), True),
                                     StructField("lng", DoubleType(), True),
                                     StructField("alt", IntegerType(), True),
                                     StructField("url", StringType(), True)
])

In [0]:
# Crear Dataframe final df_circuits_final con el esquema anterior.
df_circuits_final = spark.read\
.option('header', True)\
.schema(circuits_schema)\
.csv(ruta)

In [0]:
# Seleccionar columnas específicas.
circuits_selected_df = df_circuits_final.select(col("circuitId"), 
                                                col("circuitRef"), 
                                                col("name"), col("location"), 
                                                col("country"), 
                                                col("lat"), 
                                                col("lng"), 
                                                col("alt"))

In [0]:
# Renombrar columnas.
circuits_renamed_df = circuits_selected_df.withColumnRenamed("circuitId", "circuit_id") \
                                            .withColumnRenamed("circuitRef", "circuit_ref") \
                                            .withColumnRenamed("lat", "latitude") \
                                            .withColumnRenamed("lng", "longitude") \
                                            .withColumnRenamed("alt", "altitude") 

In [0]:
# Añadir columna ingestion_date
circuits_final_df = circuits_renamed_df.withColumn("ingestion_date", current_timestamp())

In [0]:
# Ingestar data en la tabla circuits.
circuits_final_df.write.mode("overwrite").insertInto(f"{catalogo}.{esquema}.circuits")